# E-commerce

This Colab notebook was generated from the FeatureMesh docs tutorial.

Work top to bottom: first install the FeatureMesh client, then create clients, then load data and run the tutorial.

1. **Install FeatureMesh** — Python packages for this Colab runtime.
2. **Create the BatchClient** — Jupyter magic and a local DuckDB-backed client.
3. **Follow the tutorial** — run the remaining cells in order.


## 1. Install FeatureMesh

Install the client packages for this Colab runtime.


In [ ]:
%pip install -q featuremesh pandas


## 2. Create the BatchClient

Load the Jupyter magic and create a local `BatchClient`.


In [ ]:
%load_ext featuremesh


In [ ]:
from IPython.display import display
from featuremesh import BatchClient, set_default

client = BatchClient()
set_default("client", client)
print("FeatureMesh BatchClient ready (local DuckDB)")


Map a small retail dataset — customers, orders, and items — then walk every relationship shape you’ll need later: same-entity joins, aggregations, lookups, multi-hop paths, and array enrichment with `EXTEND()`.

New to FeatureQL? The [homepage companion](https://featuremesh.com/docs/tutorials/homepage/featureql) gives the shortest SQL comparison. You can also start here with this map:

- `ENTITY()` declares the business objects; `INPUT(BIGINT#CUSTOMERS)` declares a typed customer key.
- `EXTERNAL_COLUMNS()` maps physical table columns to reusable features.
- `FROM FM.ECOMM` loads the persisted model; `FOR` binds the concrete customer, order, or item keys to evaluate.
- Features on the same entity align directly. `RELATED()` crosses entity relationships for lookups and aggregations.
- `NESTED` bindings provide keys used inside a relationship without multiplying the outer result.

Run Data and Model before the relationship examples. After a notebook restart, rerun those sections rather than executing a downstream cell in isolation.


## Data

Three customers, four orders, seven line items. Two small OBT helpers store each customer’s order-id array (plus last order) and each order’s total price. They are tutorial shortcuts that keep the array/`EXTEND()` examples focused; a production model could derive or materialize them upstream.


In [3]:
%%featureql --client client --hide-dataframe

DROP FEATURES IF EXISTS IN FM.ECOMM UP TO LEVEL 9;


*** Warning(s) (1) ***

  1. [DROP-FEATURES-EXPRESSION-EMPTY] No features found in expression: *(FEATURES IF EXISTS IN FM.ECOMM UP TO LEVEL 9) (acknowledge with ACK-ZJNH)


In [4]:
%%featureql --client client --hide-dataframe

/* SQL */
CREATE SCHEMA IF NOT EXISTS tutorial_ecomm;
--
DROP TABLE IF EXISTS tutorial_ecomm.order_totals;
--
DROP TABLE IF EXISTS tutorial_ecomm.customer_orders;
--
DROP TABLE IF EXISTS tutorial_ecomm.items;
--
DROP TABLE IF EXISTS tutorial_ecomm.orders;
--
DROP TABLE IF EXISTS tutorial_ecomm.customers;
--
CREATE TABLE tutorial_ecomm.customers (
  customer_id BIGINT,
  name VARCHAR,
  time_create TIMESTAMP
);
--
INSERT INTO tutorial_ecomm.customers VALUES
  (100, 'John Doe', TIMESTAMP '2025-09-18 14:30:00'),
  (101, 'Jane Doe', TIMESTAMP '2024-12-08 09:45:00'),
  (102, 'Jack Doe', TIMESTAMP '2025-02-03 12:00:02');
--
CREATE TABLE tutorial_ecomm.orders (
  order_id BIGINT,
  order_customer_id BIGINT,
  order_city_name VARCHAR,
  time_create TIMESTAMP
);
--
INSERT INTO tutorial_ecomm.orders VALUES
  (200, 100, 'Barcelona', TIMESTAMP '2025-09-18 15:00:00'),
  (201, 101, 'Barcelona', TIMESTAMP '2024-12-08 09:45:00'),
  (202, 102, 'Barcelona', TIMESTAMP '2025-02-03 12:00:02'),
  (203, 100, 'Madrid', TIMESTAMP '2025-09-18 14:30:00');
--
CREATE TABLE tutorial_ecomm.items (
  item_id BIGINT,
  item_order_id BIGINT,
  item_product_name VARCHAR,
  price DECIMAL(10,2),
  quantity BIGINT
);
--
INSERT INTO tutorial_ecomm.items VALUES
  (300, 200, '400', 10.05, 2),
  (301, 200, '400', 11.05, 1),
  (302, 201, '401', 12.05, 3),
  (303, 202, '402', 13.05, 2),
  (304, 202, '402', 14.05, 1),
  (305, 202, '402', 15.05, 1),
  (306, 203, '400', 16.05, 1);
--
CREATE TABLE tutorial_ecomm.customer_orders (
  customer_id BIGINT,
  last_order_id BIGINT,
  orders BIGINT[]
);
--
INSERT INTO tutorial_ecomm.customer_orders VALUES
  (100, 200, [200, 203]),
  (101, 201, [201]),
  (102, 202, [202]);
--
CREATE TABLE tutorial_ecomm.order_totals (
  order_id BIGINT,
  price DECIMAL(10,2)
);
--
INSERT INTO tutorial_ecomm.order_totals VALUES
  (200, 31.15),
  (201, 36.15),
  (202, 55.20),
  (203, 16.05);


In [5]:
%%featureql --client client

/* SQL */
SELECT customer_id, name
FROM tutorial_ecomm.customers
ORDER BY customer_id;


,customer_id,name
0,100,John Doe
1,101,Jane Doe
2,102,Jack Doe


**John** has two orders (Barcelona + Madrid). Jane and Jack have one each.

## Model

Declare entities and primary-key inputs. The `#CUSTOMERS` / `#ORDERS` / `#ITEMS` annotations are how FeatureQL knows which relationships are valid.


In [6]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.ECOMM AS
SELECT
    customers := ENTITY(),
    orders := ENTITY(),
    items := ENTITY(),
    customer_id := INPUT(BIGINT#customers),
    order_id := INPUT(BIGINT#orders),
    item_id := INPUT(BIGINT#items)
;


,feature_name,status,message
0,FM.ECOMM.CUSTOMERS,CREATED,Feature created as not exists
1,FM.ECOMM.ORDERS,CREATED,Feature created as not exists
2,FM.ECOMM.ITEMS,CREATED,Feature created as not exists
3,FM.ECOMM.CUSTOMER_ID,CREATED,Feature created as not exists
4,FM.ECOMM.ORDER_ID,CREATED,Feature created as not exists
5,FM.ECOMM.ITEM_ID,CREATED,Feature created as not exists


Map tables with `EXTERNAL_COLUMNS()`. `TABLES.*` holds raw sources; queries below compose on short names via `FROM FM.ECOMM`.


In [7]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.ECOMM AS
SELECT
    tables.customers := EXTERNAL_COLUMNS(
        customer_id BIGINT#customers BIND TO customer_id,
        name VARCHAR,
        time_create TIMESTAMP
        FROM TABLE(tutorial_ecomm.customers)
    ),
    tables.orders := EXTERNAL_COLUMNS(
        order_id BIGINT#orders BIND TO order_id,
        order_customer_id BIGINT#customers,
        order_city_name VARCHAR,
        time_create TIMESTAMP
        FROM TABLE(tutorial_ecomm.orders)
    ),
    tables.items := EXTERNAL_COLUMNS(
        item_id BIGINT#items BIND TO item_id,
        item_order_id BIGINT#orders,
        item_product_name VARCHAR,
        price DECIMAL(10,2),
        quantity BIGINT
        FROM TABLE(tutorial_ecomm.items)
    ),
    tables.customer_orders := EXTERNAL_COLUMNS(
        customer_id BIGINT#customers BIND TO customer_id,
        last_order_id BIGINT#orders,
        orders ARRAY(BIGINT#orders)
        FROM TABLE(tutorial_ecomm.customer_orders)
    ),
    tables.order_totals := EXTERNAL_COLUMNS(
        order_id BIGINT#orders BIND TO order_id,
        price DECIMAL(10,2)
        FROM TABLE(tutorial_ecomm.order_totals)
    )
;


,feature_name,status,message
0,FM.ECOMM.TABLES.CUSTOMERS,CREATED,Feature created as not exists
1,FM.ECOMM.TABLES.ORDERS,CREATED,Feature created as not exists
2,FM.ECOMM.TABLES.ITEMS,CREATED,Feature created as not exists
3,FM.ECOMM.TABLES.CUSTOMER_ORDERS,CREATED,Feature created as not exists
4,FM.ECOMM.TABLES.ORDER_TOTALS,CREATED,Feature created as not exists


## Peek at the bindings

Bind customer ids → dimension rows. Bind order ids → order facts.


In [8]:
%%featureql --client client

SELECT
    CUSTOMER_ID,
    NAME := TABLES.CUSTOMERS[name],
    TIME_CREATE := TABLES.CUSTOMERS[time_create]
FROM FM.ECOMM
FOR
    CUSTOMER_ID := BIND_COLUMNS(customer_id FROM TABLE(tutorial_ecomm.customers))
ORDER BY CUSTOMER_ID;


,FM.ECOMM.CUSTOMER_ID,NAME,TIME_CREATE
0,100,John Doe,2025-09-18 14:30:00
1,101,Jane Doe,2024-12-08 09:45:00
2,102,Jack Doe,2025-02-03 12:00:02


In [9]:
%%featureql --client client

SELECT
    ORDER_ID,
    ORDER_CUSTOMER_ID := TABLES.ORDERS[order_customer_id],
    CITY := TABLES.ORDERS[order_city_name],
    TIME_CREATE := TABLES.ORDERS[time_create]
FROM FM.ECOMM
FOR
    ORDER_ID := BIND_COLUMNS(order_id FROM TABLE(tutorial_ecomm.orders))
ORDER BY ORDER_ID;


,FM.ECOMM.ORDER_ID,ORDER_CUSTOMER_ID,CITY,TIME_CREATE
0,200,100,Barcelona,2025-09-18 15:00:00
1,201,101,Barcelona,2024-12-08 09:45:00
2,202,102,Barcelona,2025-02-03 12:00:02
3,203,100,Madrid,2025-09-18 14:30:00


## Same-entity join (PK → PK)

Features that share `CUSTOMER_ID` align automatically — no `RELATED()` needed.


In [10]:
%%featureql --client client

SELECT
    CUSTOMER_ID,
    CUSTOMER_NAME := TABLES.CUSTOMERS[name],
    CUSTOMER_ORDERS := TABLES.CUSTOMER_ORDERS[orders]
FROM FM.ECOMM
FOR
    CUSTOMER_ID := BIND_VALUES(ARRAY[100, 101, 102])
ORDER BY CUSTOMER_ID;


,FM.ECOMM.CUSTOMER_ID,CUSTOMER_NAME,CUSTOMER_ORDERS
0,100,John Doe,"[200, 203]"
1,101,Jane Doe,[201]
2,102,Jack Doe,[202]


John → orders **[200, 203]**.

## Aggregation (PK → FK)

Count orders per customer: aggregate on the FK, join back to the PK. Replaces a GROUP BY subquery + LEFT JOIN.


In [11]:
%%featureql --client client

SELECT
    CUSTOMER_ID,
    NUM_ORDERS := RELATED(SUM(1) GROUP BY TABLES.ORDERS[order_customer_id] VIA CUSTOMER_ID)
FROM FM.ECOMM
FOR
    CUSTOMER_ID := BIND_VALUES(ARRAY[100, 101, 102]),
    NESTED ORDER_ID := BIND_VALUES(ARRAY[200, 201, 202, 203])
ORDER BY CUSTOMER_ID;


,FM.ECOMM.CUSTOMER_ID,NUM_ORDERS
0,100,2
1,101,1
2,102,1


John **2**, Jane **1**, Jack **1**.

## Lookup (FK → PK)

Follow `last_order_id` to the order’s city.


In [12]:
%%featureql --client client

WITH
    LAST_ORDER_CITY := RELATED(TABLES.ORDERS[order_city_name] VIA TABLES.CUSTOMER_ORDERS[last_order_id])
SELECT
    CUSTOMER_ID,
    LAST_ORDER_CITY
FROM FM.ECOMM
FOR
    CUSTOMER_ID := BIND_VALUES(ARRAY[100, 101, 102])
ORDER BY CUSTOMER_ID;


,FM.ECOMM.CUSTOMER_ID,LAST_ORDER_CITY
0,100,Barcelona
1,101,Barcelona
2,102,Barcelona


All three current last orders are in **Barcelona**.

## Multi-hop (FK → FK)

From the customer’s last order, sum item `price × quantity` on that order.


In [13]:
%%featureql --client client

WITH
    LAST_ORDER_ID := TABLES.CUSTOMER_ORDERS[last_order_id],
    LAST_ORDER_PRICE := RELATED(
        SUM(TABLES.ITEMS[price] * TABLES.ITEMS[quantity]::DECIMAL) GROUP BY TABLES.ITEMS[item_order_id]
        VIA LAST_ORDER_ID
    )
SELECT
    CUSTOMER_ID,
    LAST_ORDER_ID,
    LAST_ORDER_PRICE
FROM FM.ECOMM
FOR
    CUSTOMER_ID := BIND_VALUES(ARRAY[100, 101, 102]),
    NESTED ITEM_ID := BIND_VALUES(ARRAY[300, 301, 302, 303, 304, 305, 306])
ORDER BY CUSTOMER_ID;


,FM.ECOMM.CUSTOMER_ID,LAST_ORDER_ID,LAST_ORDER_PRICE
0,100,200,31.15
1,101,201,36.15
2,102,202,55.20


John’s last order **200** → **31.15**. Jane **36.15**. Jack **55.20**.

## Enrich arrays with `EXTEND()`

`ZIP` the order-id array into rows, then look up each order’s total price.


In [14]:
%%featureql --client client

SELECT
    CUSTOMER_ID,
    CUSTOMER_ORDERS := TABLES.CUSTOMER_ORDERS[orders],
    CUSTOMER_ORDERS_DETAILS := EXTEND(
        ZIP(CUSTOMER_ORDERS AS order_id)
        WITH TABLES.ORDER_TOTALS[price] AS ORDER_PRICE
    )
FROM FM.ECOMM
FOR
    CUSTOMER_ID := BIND_VALUES(ARRAY[100, 101, 102])
ORDER BY CUSTOMER_ID;


,FM.ECOMM.CUSTOMER_ID,CUSTOMER_ORDERS,CUSTOMER_ORDERS_DETAILS
0,100,"[200, 203]","[{'order_id': 200, 'order_price': 31.15}, {'or..."
1,101,[201],"[{'order_id': 201, 'order_price': 36.15}]"
2,102,[202],"[{'order_id': 202, 'order_price': 55.20}]"


John → order **200** at **31.15**, order **203** at **16.05**.

## All four patterns together

One query: name + order array, last-order city, order count, last-order price. Bind every entity that appears in the dependency graph (`NESTED` for orders and items).


In [15]:
%%featureql --client client

WITH
    CUSTOMER_NAME := TABLES.CUSTOMERS[name],
    CUSTOMER_ORDERS := TABLES.CUSTOMER_ORDERS[orders],
    LAST_ORDER_ID := TABLES.CUSTOMER_ORDERS[last_order_id],
    LAST_ORDER_CITY := LAST_ORDER_ID.RELATED(TABLES.ORDERS[order_city_name]),
    NUM_ORDERS := CUSTOMER_ID.RELATED(SUM(1) GROUP BY TABLES.ORDERS[order_customer_id]),
    LAST_ORDER_PRICE := LAST_ORDER_ID.RELATED(
        SUM(TABLES.ITEMS[price] * TABLES.ITEMS[quantity]::DECIMAL)
        GROUP BY TABLES.ITEMS[item_order_id]
    )
SELECT
    CUSTOMER_ID,
    CUSTOMER_NAME,
    CUSTOMER_ORDERS,
    LAST_ORDER_ID,
    LAST_ORDER_CITY,
    NUM_ORDERS,
    LAST_ORDER_PRICE
FROM FM.ECOMM
FOR
    CUSTOMER_ID := BIND_VALUES(ARRAY[100, 101, 102]),
    NESTED ORDER_ID := BIND_VALUES(ARRAY[200, 201, 202, 203]),
    NESTED ITEM_ID := BIND_VALUES(ARRAY[300, 301, 302, 303, 304, 305, 306])
ORDER BY CUSTOMER_ID;


,FM.ECOMM.CUSTOMER_ID,CUSTOMER_NAME,CUSTOMER_ORDERS,LAST_ORDER_ID,LAST_ORDER_CITY,NUM_ORDERS,LAST_ORDER_PRICE
0,100,John Doe,"[200, 203]",200,Barcelona,2,31.15
1,101,Jane Doe,[201],201,Barcelona,1,36.15
2,102,Jack Doe,[202],202,Barcelona,1,55.20


## What's next

- [Analytics overview](https://featuremesh.com/docs/tutorials/analytics/overview) — concept map for this series
- [SaaS metrics](https://featuremesh.com/docs/tutorials/analytics/saas) — MRR and customer health on a persisted model
- [OBT modeling](https://featuremesh.com/docs/tutorials/analytics/obt) — deeper `ARRAY(ROW)` work
- [RELATED](https://featuremesh.com/docs/featureql/structural_operations/related) / [EXTEND](https://featuremesh.com/docs/featureql/array_of_rows/extend) — full syntax


---

Source tutorial: [/docs/tutorials/analytics/ecomm](https://featuremesh.com/docs/tutorials/analytics/ecomm)
